In [2]:
#torchvision has pretrained model builtin like ResNet, MobileNet, etc. We can 
#load them and save the weights to see how big they are in FP32, and estimate the size after quantization to INT8. This is important for automotive ECUs where memory is limited.

import torchvision.models as models
import torch
import os

# Load MobileNetV2 pretrained on ImageNet
# MobileNet is designed for mobile/edge deployment — smaller than ResNet
# But even "small" models are too big for automotive ECUs without quantization
model = models.mobilenet_v2(weights='IMAGENET1K_V1') #load pretrained MobileNetV2
# a model with 3.4 million parameters, which is relatively small compared to larger models like ResNet50 (25 million) or ResNet101 (44 million). However, even 3.4 million parameters can be too large for automotive ECUs without quantization.
# It is pretrained on ImageNet: ImageNet is a large dataset of 1.2 million images across categories — cats, dogs, cars, planes, furniture, food, animals, tools. Everything.
# ImageNet is commonly used for training and benchmarking image classification models. Pretraining on ImageNet allows the model to learn general features that can be useful for various computer vision tasks, including those relevant to automotive applications (e.g., object detection, lane detection, etc.). However, the pretrained weights are typically in FP32 format, which can be too
#  large for deployment on resource-constrained automotive ECUs without quantization.
model.eval()

# Save FP32 weights: FP32 bits each number is stored in 32 bit Floating point value 
# Why we take floating point is because floating point keeps the decimal precision
# If we take INT8, it wll be 8 bits per value, which is 4 times smaller than FP32 (32 bits).
# # So we can expect the size to be reduced by a factor of 4 when quantized to INT8.
# but question is how to handle precise value as we need in training 
# Because Precision matter less during the inference because we are not doing gradient updates, and model already has been learned small precision change will not impact dramatically 

torch.save(model.state_dict(), 'mobilenet_fp32.pth')

size_mb = os.path.getsize('mobilenet_fp32.pth') / (1024 * 1024)
print(f"MobileNetV2 FP32 size : {size_mb:.1f} MB")
print(f"Weight precision      : float32 = 4 bytes per value")
print(f"Expected INT8 size    : ~{size_mb/4:.1f} MB (4x smaller)")

MobileNetV2 FP32 size : 13.6 MB
Weight precision      : float32 = 4 bytes per value
Expected INT8 size    : ~3.4 MB (4x smaller)


### Concrete memory Picture of FP32 and INT8 Quantization
FP32: 3.4M * 4 bytes = 13.6 MB
INT8: 3.4M * 1 byte = 3.4 MB--> target 
1. Because ECU is having limited memory space
2. Because we want to reduce the model size for faster loading and inference on edge devices.
3. INT8 arithmetic is simpler than float32 arithmetic, NPU executes it 2-4x faster and at low power
4. ### This is what exactly TensorRT does before deploying the model on edge devices (Nvidia DLA on DriveOS)--> converts F32  training weights to INT8 for production inference 


In [4]:
# CELL 2 — Apply Dynamic Quantization (INT8)
# Dynamic quantization converts weights from float32 to int8
# "Dynamic" means activations are quantized at runtime, not in advance
# This is the simplest form of quantization — no calibration data needed

torch.backends.quantized.engine = 'qnnpack'

quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

torch.save(quantized_model.state_dict(), 'mobilenet_int8.pth')

size_fp32 = os.path.getsize('mobilenet_fp32.pth') / (1024 * 1024)  # read actual file
size_int8 = os.path.getsize('mobilenet_int8.pth') / (1024 * 1024)
reduction = size_fp32 / size_int8

print(f"MobileNetV2 FP32 size : {size_fp32:.1f} MB")
print(f"MobileNetV2 INT8 size : {size_int8:.1f} MB")
print(f"Size reduction        : {reduction:.1f}x smaller")
print(f"Space saved           : {size_fp32 - size_int8:.1f} MB")

MobileNetV2 FP32 size : 13.6 MB
MobileNetV2 INT8 size : 9.9 MB
Size reduction        : 1.4x smaller
Space saved           : 3.7 MB


/var/folders/4b/l78vh3lx7xnbl5rwg4ph0n3w0000gn/T/ipykernel_3568/1647272189.py:8: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Our Target was to reduce about 4 times but the mode size is reduced by 1.4x to 3.7MB only dynamic quantization only quantized the linear layers and not the convolutional layers, which are the majority of the parameters in MobileNetV2.
To get the target of reduction 4x I need Static quantization which quantizes Con layer too 



### Dynamic and Static Quantization
- Dynamic quantization: Only weights are quantized to INT8, activations are quantized at runtime. No preparation needed, but less efficient.
- Static quantization: Both weights and activations are quantized to INT8. Requires calibration.
- Conv layers need calibration because their activation values vary depending on the input images. 
- We need to know the real data to know the typical range.
-- NVidia TensorRT uses static quantization for best performance on edge devices, before deploying the model on Nvidia DLA (DriveOS) it converts the FP32 training weights to INT8 for production inference.
-- They run the calibration images through the model, measure activation ranges, then freeze  activation ranges then freeze the INT8 ranges.
-- The deployed model on the car is fully static deterministic, fast,predictable latency, and low power.
-- In OTA — you don't push raw firmware to 10,000 vehicles without validation. You test on a sample first. Check behaviour. Confirm it works as expected. Then deploy.
-- Static quantization does the same thing:
-- You don't convert the model blindly. You run ~100 calibration images through it first. The model observes real data, measures the activation ranges in each Conv layer, then freezes those ranges for INT8 conversion. Validation before deployment.
-- Dynamic quantization = pushing firmware without testing. Works for simple cases, but you're guessing the ranges.
-- When an image passes through a Conv layer, it produces output values. Maybe they range from -3.2 to +8.7 for cat images. For dog images, -2.1 to +6.4. Across 100 calibration images, the typical range is roughly -4 to +9.
-- Static quantization records this range per layer. Then maps that entire range to INT8 values -128 to +127. Clean, informed conversion.
-- Without calibration — you're guessing the range. If the actual values go outside your guessed range, they get clipped. Information lost. Accuracy drops.

In [6]:
# CELL 3 — Static Quantization (Conv + Linear layers)
# Needs calibration data to measure activation ranges


import copy
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Step 1: Prepare model for static quantization
# QuantStub/DeQuantStub mark where quantization starts and ends
static_model = copy.deepcopy(model)  # Don't modify original FP32 model
static_model.eval()

static_model.qconfig = torch.quantization.get_default_qconfig('qnnpack')
torch.quantization.prepare(static_model, inplace=True)

# Step 2: Calibration — run real images through model
# Model observes activation ranges per layer — no training, just watching
print("Calibrating with real images...")

calibration_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Use your existing cat/dog/bird dataset for calibration
calibration_data = datasets.ImageFolder(
    root='../image_classifier/data',

    transform=calibration_transform
)
calibration_loader = DataLoader(
    calibration_data, batch_size=16, shuffle=True
)

# Run ~100 images through — model records activation ranges
static_model.eval()
with torch.no_grad():
    for i, (images, _) in enumerate(calibration_loader):
        static_model(images)
        if i >= 6:  # ~100 images (16 per batch × 6 batches)
            break

print("Calibration complete.")

# Step 3: Convert to INT8 — now Conv layers are quantized too
torch.quantization.convert(static_model, inplace=True)

# Step 4: Measure size
torch.save(static_model.state_dict(), 'mobilenet_static_int8.pth')
size_static = os.path.getsize('mobilenet_static_int8.pth') / (1024 * 1024)
reduction_static = size_fp32 / size_static

print(f"\nMobileNetV2 FP32         : {size_fp32:.1f} MB")
print(f"MobileNetV2 Dynamic INT8 : {size_int8:.1f} MB (1.4x)")
print(f"MobileNetV2 Static INT8  : {size_static:.1f} MB ({reduction_static:.1f}x)")
print(f"\nConv layers now quantized — that's where the real saving is.")

/var/folders/4b/l78vh3lx7xnbl5rwg4ph0n3w0000gn/T/ipykernel_3568/1567132912.py:15: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare(static_model, inplace=True)


Calibrating with real images...
Calibration complete.


/var/folders/4b/l78vh3lx7xnbl5rwg4ph0n3w0000gn/T/ipykernel_3568/1567132912.py:49: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.convert(static_model, inplace=True)



MobileNetV2 FP32         : 13.6 MB
MobileNetV2 Dynamic INT8 : 9.9 MB (1.4x)
MobileNetV2 Static INT8  : 3.8 MB (3.6x)

Conv layers now quantized — that's where the real saving is.
